In [340]:
# ============================================================
# GOAT-Net — Cell A: Rich Metadata Schema Configuration
# ============================================================
import os
import sys
import subprocess

# Install RapidFuzz if needed
try:
    import rapidfuzz
except ImportError:
    print("⏳ Installing rapidfuzz for high-speed name clustering...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rapidfuzz"])
    import rapidfuzz

import yaml
from pathlib import Path
from google.colab import drive

# Mount Drive & set working directory
drive.mount("/content/drive", force_remount=False)
REPO = Path("/content/drive/MyDrive/GOAT-Net")
os.chdir(REPO)

CONFIG_DIR = REPO / "config"
CONFIG_DIR.mkdir(parents=True, exist_ok=True)
SCHEMA_PATH = CONFIG_DIR / "dataset_schema.yml"

schema_content = """
pipeline_version: "3.5.0"
cutoff_date: "2026-07-31"

datasets:
  statistical_player_summary:
    type: "statistical"
    path: "data/processed/statistical/player_summary.parquet"
    provider: "fbref"
    granularity: "career_summary"
    keys:
      primary_player: "common_name"
      match_count: "matches"

  statistical_season_stats:
    type: "statistical"
    path: "data/processed/statistical/player_season_stats.parquet"
    provider: "fbref"
    granularity: "player_season"
    keys:
      primary_player: "common_name"
      season: "season"
      competition: "comp"
      match_count: "matches"

  statistical_match_stats:
    type: "statistical"
    path: "data/processed/statistical/player_match_stats.parquet"
    provider: "fbref"
    granularity: "player_match"
    keys:
      primary_player: "common_name"
      match: "match_id"
      season: "season"
      competition: "comp"
      date: "date"

  spatial_player_summary:
    type: "spatial"
    path: "data/processed/spatial/player_spatial_summary.parquet"
    provider: "statsbomb"
    granularity: "career_summary"
    keys:
      primary_player: "common_name"
      secondary_players: ["statsbomb_player"]
      match_count: "matches"

  spatial_passing_centrality:
    type: "spatial"
    path: "data/processed/spatial/match_passing_centrality.parquet"
    provider: "statsbomb"
    granularity: "player_match"
    keys:
      primary_player: "common_name"
      secondary_players: ["statsbomb_player"]
      match: "match_id"

  spatial_zone_summary:
    type: "spatial"
    path: "data/processed/spatial/player_zone_summary.parquet"
    provider: "statsbomb"
    granularity: "career_summary"
    keys:
      primary_player: "common_name"
      secondary_players: ["statsbomb_player"]
"""

with open(SCHEMA_PATH, "w") as f:
    f.write(schema_content.strip())

print(f"✅ Rich Configuration schema saved: {SCHEMA_PATH}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Rich Configuration schema saved: /content/drive/MyDrive/GOAT-Net/config/dataset_schema.yml


In [341]:
# ============================================================
# GOAT-Net — Cell A.5: Create aliases.yml
# ============================================================
import yaml
from pathlib import Path

CONFIG_DIR = Path("/content/drive/MyDrive/GOAT-Net/config")
CONFIG_DIR.mkdir(parents=True, exist_ok=True)
ALIASES_PATH = CONFIG_DIR / "aliases.yml"

aliases_content = {
    "Lionel Andrés Messi Cuccittini": {
        "preferred_name": "Lionel Messi",
        "aliases": ["L. Messi", "Lionel Messi"]
    },
    "Cristiano Ronaldo dos Santos Aveiro": {
        "preferred_name": "Cristiano Ronaldo",
        "aliases": ["C. Ronaldo", "Cristiano Ronaldo"]
    },
    "Neymar da Silva Santos Junior": {
        "preferred_name": "Neymar",
        "aliases": ["Neymar Jr", "Neymar"]
    },
    "Kylian Mbappé Lottin": {
        "preferred_name": "Kylian Mbappé",
        "aliases": ["K. Mbappé", "Kylian Mbappé"]
    }
}

with open(ALIASES_PATH, "w") as f:
    yaml.dump(aliases_content, f, allow_unicode=True, sort_keys=False)

print(f"✅ Created aliases configuration: {ALIASES_PATH}")

✅ Created aliases configuration: /content/drive/MyDrive/GOAT-Net/config/aliases.yml


In [342]:
# # ============================================================
# # GOAT-Net — Cell B: Automated Extraction, Normalization & Clustering
# # ============================================================
# import re
# import yaml
# import unicodedata
# import pandas as pd
# from datetime import datetime
# from rapidfuzz import fuzz, process

# # 1. Text Normalization Helper
# def normalize_name(name_str):
#     """Strips accents/diacritics, lowercases, and removes punctuation."""
#     if not isinstance(name_str, str) or not name_str.strip():
#         return ""
#     # Strip diacritical marks (e.g. Ó, í, ć -> O, i, c)
#     text = unicodedata.normalize("NFD", name_str).encode("ascii", "ignore").decode("utf-8")
#     text = text.lower().strip()
#     # Remove special characters/punctuation except spaces
#     text = re.sub(r"[^a-z0-9\s]", "", text)
#     text = re.sub(r"\s+", " ", text)
#     return text

# # 2. Load Configuration
# SCHEMA_PATH = CONFIG_DIR / "dataset_schema.yml"
# with open(SCHEMA_PATH, "r") as f:
#     schema_config = yaml.safe_load(f)

# datasets_schema = schema_config.get("datasets", {})
# pipeline_version = schema_config.get("pipeline_version", "3.5.0")

# print("⚙️ Ingesting datasets and extracting player records...")

# raw_records = []
# for dataset_key, config in datasets_schema.items():
#     file_path = REPO / config["path"]
#     if not file_path.exists():
#         raise FileNotFoundError(f"❌ Missing required file: {file_path}")

#     df = pd.read_parquet(file_path)
#     for col in config["player_columns"]:
#         if col in df.columns:
#             counts = df[col].dropna().value_counts()
#             for name, freq in counts.items():
#                 name_clean = str(name).strip()
#                 if name_clean and name_clean.lower() != "nan":
#                     raw_records.append({
#                         "source_dataset": dataset_key,
#                         "source_column": col,
#                         "source_player_name": name_clean,
#                         "normalized_name": normalize_name(name_clean),
#                         "source_record_count": int(freq)
#                     })

# df_raw = pd.DataFrame(raw_records)

# # Aggregate record counts per unique raw name variant
# df_variants = (
#     df_raw.groupby(["source_player_name", "normalized_name"], as_index=False)["source_record_count"]
#     .sum()
#     .sort_values(by="source_record_count", ascending=False)
# )

# print(f"📊 Discovered {len(df_variants)} unique raw name variants.")

# # 3. Automated Clustering Engine (RapidFuzz)
# print("🤖 Auto-clustering player variants using RapidFuzz...")

# unique_normalized = df_variants["normalized_name"].unique().tolist()
# assigned_groups = {}
# cluster_counter = 1

# # Process high-frequency names first
# for norm_name in unique_normalized:
#     if not norm_name or norm_name in assigned_groups:
#         continue

#     cluster_id = f"GROUP_{cluster_counter:05d}"
#     assigned_groups[norm_name] = cluster_id

#     # Find all unassigned fuzzy matches (Token-Set Ratio >= 88 handles word order differences)
#     unassigned = [n for n in unique_normalized if n not in assigned_groups]
#     if unassigned:
#         matches = process.extract(
#             norm_name,
#             unassigned,
#             scorer=fuzz.token_set_ratio,
#             score_cutoff=88.0,
#             limit=None
#         )
#         for match_norm, score, _ in matches:
#             assigned_groups[match_norm] = cluster_id

#     cluster_counter += 1

# df_variants["cluster_id"] = df_variants["normalized_name"].map(assigned_groups)

# # Determine Suggested Canonical Name per Cluster (Variant with highest record count)
# top_variant_per_cluster = (
#     df_variants.sort_values(by="source_record_count", ascending=False)
#     .groupby("cluster_id")["source_player_name"]
#     .first()
#     .to_dict()
# )

# df_variants["suggested_canonical_name"] = df_variants["cluster_id"].map(top_variant_per_cluster)

# # Merge back into full dataset
# df_final_template = df_raw.merge(
#     df_variants[["source_player_name", "cluster_id", "suggested_canonical_name"]],
#     on="source_player_name",
#     how="left"
# )

# # Classify Tiers based on Match Quality
# def assign_tier(row):
#     if row["source_player_name"] == row["suggested_canonical_name"]:
#         return "Tier 1 - Exact Match"
#     elif row["normalized_name"] == normalize_name(row["suggested_canonical_name"]):
#         return "Tier 2 - Accent/Case Match"
#     else:
#         return "Tier 3 - Fuzzy Cluster Match"

# df_final_template["confidence_tier"] = df_final_template.apply(assign_tier, axis=1)

# # Default 'canonical_name' to 'suggested_canonical_name' (Human review is edit-by-exception)
# df_final_template["canonical_name"] = df_final_template["suggested_canonical_name"]
# df_final_template["notes"] = ""

# # Sort by cluster_id and record count
# df_final_template = df_final_template.sort_values(
#     by=["source_record_count", "cluster_id"], ascending=[False, True]
# )

# # 4. Save Outputs
# CANDIDATE_PATH = CONFIG_DIR / "candidate_duplicates.csv"
# REVIEW_PATH = CONFIG_DIR / "review_required.csv"

# df_final_template.to_csv(CANDIDATE_PATH, index=False)

# # Filter Tier 3 items that might need quick human verification
# df_review = df_final_template[df_final_template["confidence_tier"] == "Tier 3 - Fuzzy Cluster Match"]
# df_review.to_csv(REVIEW_PATH, index=False)

# print("\n" + "="*70)
# print(f"🎉 CLUSTERING COMPLETE: {df_final_template['cluster_id'].nunique()} Candidate Player Groups Formed.")
# print(f"💾 Full Candidate Ledger: {CANDIDATE_PATH}")
# print(f"⚠️ Uncertain Groups Requiring Review: {len(df_review)} rows ({REVIEW_PATH})")
# print("="*70)
# print("\n💡 NOTE:")
# print("95%+ of names were automatically grouped and auto-filled.")
# print("If you approve the auto-generated clusters, you can run Cell C directly!")

In [343]:
# ============================================================
# GOAT-Net — Cell B: Automated Extraction, Normalization & STRICT Clustering
# FINAL VERSION – Pre-merged aliases, dual-token blocking, fixed normalization
# (UPDATED: stricter fuzzy matching, surname checks, refined blocking)
# ============================================================
import re
import yaml
import unicodedata
import pandas as pd
from pathlib import Path
from rapidfuzz import fuzz
from itertools import combinations
from collections import defaultdict

import re
import unicodedata
import pandas as pd
from rapidfuzz import fuzz
from itertools import combinations
from collections import defaultdict

# ----------------------------------------------------------------------
# 0. Configuration (adjust these paths to your environment)
# ----------------------------------------------------------------------
REPO = Path(".")        # root of your repository
CONFIG_DIR = REPO / "config"  # directory containing aliases.yml & dataset_schema.yml

# ----------------------------------------------------------------------
# 1. Union-Find (Disjoint Set) – Transitive Clustering
# ----------------------------------------------------------------------
class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, x, y):
        rx, ry = self.find(x), self.find(y)
        if rx == ry:
            return
        if self.rank[rx] < self.rank[ry]:
            self.parent[rx] = ry
        elif self.rank[ry] < self.rank[rx]:
            self.parent[ry] = rx
        else:
            self.parent[ry] = rx
            self.rank[rx] += 1

# ----------------------------------------------------------------------
# 2. Text Normalization Helper
# ----------------------------------------------------------------------
def normalize_name(name_str):
    """Strips accents/diacritics, lowercases, removes apostrophes and punctuation."""
    if not isinstance(name_str, str) or not name_str.strip():
        return ""
    text = unicodedata.normalize("NFD", name_str).encode("ascii", "ignore").decode("utf-8")
    text = text.lower().strip()
    text = text.replace("’", "'")
    text = re.sub(r"'+", "'", text)
    text = re.sub(r"[^a-z0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text

# ----------------------------------------------------------------------
# 3. Load External Alias Mapping (richer structure supported)
# ----------------------------------------------------------------------
ALIASES_PATH = CONFIG_DIR / "aliases.yml"

alias_to_canonical = {} # normalised alias -> normalised canonical
alias_to_preferred = {} # normalised alias -> original preferred name string

if ALIASES_PATH.exists():
    with open(ALIASES_PATH, "r") as f:
        raw_aliases = yaml.safe_load(f) or {}

    for full_name, entry in raw_aliases.items():
        if isinstance(entry, list):
            canonical_display = full_name
            aliases = entry
        elif isinstance(entry, dict):
            canonical_display = entry.get("preferred_name", full_name)
            aliases = entry.get("aliases", [])
        else:
            continue

        canonical_norm = normalize_name(canonical_display)

        variants = [full_name] + aliases
        for variant in variants:
            v_norm = normalize_name(variant)
            alias_to_canonical[v_norm] = canonical_norm
            alias_to_preferred[v_norm] = canonical_display

        alias_to_canonical[canonical_norm] = canonical_norm
        alias_to_preferred[canonical_norm] = canonical_display

    print(f"📋 Loaded {len(raw_aliases)} canonical entries with aliases from aliases.yml")
else:
    print("⚠️ aliases.yml not found – alias matching will be skipped.")

# ----------------------------------------------------------------------
# 4. Unified Match Engine – STRICT matching logic
# ----------------------------------------------------------------------
MATCH_THRESHOLD = 95

def match_pair(name1, name2):
    if not name1 or not name2:
        return False, "Invalid", 0, False

    canonical1 = alias_to_canonical.get(name1)
    canonical2 = alias_to_canonical.get(name2)
    if canonical1 and canonical2 and canonical1 == canonical2:
        return True, "Alias", 100, True

    n1_words, n2_words = name1.split(), name2.split()
    first1, first2 = n1_words[0], n2_words[0]

    first_match = (
        (first1 == first2) or
        (first1.startswith(first2) and len(first2) == 1) or
        (first2.startswith(first1) and len(first1) == 1)
    )

    # FIXED ratio calculation
    ratio = max(
        fuzz.ratio(name1, name2),
        fuzz.token_sort_ratio(name1, name2)
    )

    # token_set_ratio ONLY when one string contains the other verbatim
    if len(n1_words) != len(n2_words) and (name1 in name2 or name2 in name1):
        ratio = max(ratio, fuzz.token_set_ratio(name1, name2))

    if ratio < MATCH_THRESHOLD:
        return False, "None", ratio, first_match

    # Reject if first names do not match (and both are longer than 1 char)
    if not first_match and len(first1) > 1 and len(first2) > 1:
        return False, "Fuzzy", ratio, first_match

    # Surname check
    if len(n1_words) >= 2 and len(n2_words) >= 2:
        last1, last2 = n1_words[-1], n2_words[-1]
        if last1 != last2:
            return False, "Fuzzy", ratio, first_match

    # Shared token minimum for longer names
    if len(n1_words) >= 3 and len(n2_words) >= 3:
        common = set(n1_words) & set(n2_words)
        if len(common) < 2:
            return False, "Fuzzy", ratio, first_match

    return True, "Fuzzy", ratio, first_match

def is_valid_match(name1, name2):
    matched, _, _, _ = match_pair(name1, name2)
    return matched

# ----------------------------------------------------------------------
# 5. Blocking strategy – first token + last TWO tokens
# ----------------------------------------------------------------------
def get_candidate_pairs(normalized_names):
    first_token_blocks = defaultdict(list)
    last_two_blocks = defaultdict(list)

    for idx, name in enumerate(normalized_names):
        if not name:
            continue
        tokens = name.split()
        first_token_blocks[tokens[0]].append((idx, name))
        if len(tokens) >= 2:
            last_two_key = " ".join(tokens[-2:])
        else:
            last_two_key = tokens[0]
        last_two_blocks[last_two_key].append((idx, name))

    seen_pairs = set()

    for blocks in (first_token_blocks, last_two_blocks):
        for group in blocks.values():
            if len(group) < 2:
                continue
            for (i1, n1), (i2, n2) in combinations(group, 2):
                pair = (min(i1, i2), max(i1, i2))
                if pair not in seen_pairs:
                    seen_pairs.add(pair)
                    yield i1, n1, i2, n2

# ----------------------------------------------------------------------
# 6. Load Dataset Schema Configuration
# ----------------------------------------------------------------------
SCHEMA_PATH = CONFIG_DIR / "dataset_schema.yml"
with open(SCHEMA_PATH, "r") as f:
    schema_config = yaml.safe_load(f)

datasets_schema = schema_config.get("datasets", {})
pipeline_version = schema_config.get("pipeline_version", "3.5.0")

# ----------------------------------------------------------------------
# 7. Ingest and Extract Raw Player Records
# ----------------------------------------------------------------------
print("⚙️ Ingesting datasets and extracting player records...")
raw_records = []

for dataset_key, config in datasets_schema.items():
    file_path = REPO / config["path"]
    if not file_path.exists():
        raise FileNotFoundError(f"❌ Missing required file: {file_path}")

    df = pd.read_parquet(file_path)
    # Use the new schema: collect all player columns (primary + secondary)
    player_columns = []
    keys = config.get("keys", {})
    prim = keys.get("primary_player")
    if prim:
        player_columns.append(prim)
    sec = keys.get("secondary_players", [])
    player_columns.extend(sec)

    for col in player_columns:
        if col in df.columns:
            counts = df[col].dropna().value_counts()
            for name, freq in counts.items():
                name_clean = str(name).strip()
                if name_clean and name_clean.lower() not in ["nan", "none"]:
                    raw_records.append({
                        "source_dataset": dataset_key,
                        "source_column": col,
                        "source_player_name": name_clean,
                        "normalized_name": normalize_name(name_clean),
                        "source_record_count": int(freq)
                    })

df_raw = pd.DataFrame(raw_records)

df_variants = (
    df_raw.groupby(["source_player_name", "normalized_name"], as_index=False)["source_record_count"]
    .sum()
    .sort_values(by="source_record_count", ascending=False)
)

print(f"📊 Discovered {len(df_variants)} unique raw name variants.")

# ----------------------------------------------------------------------
# 8. Union-Find Clustering with Pre-merged Aliases & Improved Blocking
# ----------------------------------------------------------------------
print("🤖 Auto-clustering player variants using Union-Find...")

unique_norms = df_variants["normalized_name"].unique().tolist()
n = len(unique_norms)
uf = UnionFind(n)
norm_to_idx = {norm: i for i, norm in enumerate(unique_norms)}

if alias_to_canonical:
    print("🔗 Pre-merging known aliases...")
    for alias_norm, canonical_norm in alias_to_canonical.items():
        if alias_norm in norm_to_idx and canonical_norm in norm_to_idx:
            uf.union(norm_to_idx[alias_norm], norm_to_idx[canonical_norm])

comparisons_done = 0
for i1, n1, i2, n2 in get_candidate_pairs(unique_norms):
    if uf.find(i1) == uf.find(i2):
        continue
    if abs(len(n1) - len(n2)) > 10:
        continue
    matched, method, score, _ = match_pair(n1, n2)
    if matched:
        uf.union(i1, i2)
    comparisons_done += 1

print(f"⚖️ Compared {comparisons_done} candidate pairs using dual-token blocking.")

# Assign cluster IDs
idx_to_cluster = {}
cluster_counter = 1
for i in range(n):
    root = uf.find(i)
    if root not in idx_to_cluster:
        idx_to_cluster[root] = f"GROUP_{cluster_counter:05d}"
        cluster_counter += 1

df_variants["cluster_id"] = df_variants["normalized_name"].map(
    lambda nm: idx_to_cluster[uf.find(norm_to_idx[nm])]
)

# ----------------------------------------------------------------------
# 9. Canonical Name Selection – alias preferred_name ALWAYS wins
# ----------------------------------------------------------------------
def select_canonical(cluster_df):
    norms = cluster_df["normalized_name"].unique()
    for norm in norms:
        if norm in alias_to_preferred:
            return alias_to_preferred[norm]
    return (
        cluster_df
        .groupby("source_player_name")["source_record_count"]
        .sum()
        .idxmax()
    )

canonical_map = {}
for cid, grp in df_variants.groupby("cluster_id"):
    canonical_map[cid] = select_canonical(grp)

df_variants["suggested_canonical_name"] = df_variants["cluster_id"].map(canonical_map)

# Merge back to full ledger
df_final_template = df_raw.merge(
    df_variants[["source_player_name", "cluster_id", "suggested_canonical_name"]],
    on="source_player_name",
    how="left"
)

# ----------------------------------------------------------------------
# 10. Compute Reason, Method, Scores
# ----------------------------------------------------------------------
def compute_reason_and_method(row):
    src = row["source_player_name"]
    can = row["suggested_canonical_name"]
    src_norm = row["normalized_name"]
    can_norm = normalize_name(can)

    if src == can:
        return "Exact", "Self", 100, True
    if src_norm == can_norm:
        return "Accent", "Accent", 100, True

    matched, method, fuzz_ratio, first_match = match_pair(src_norm, can_norm)
    if matched and method == "Alias":
        return "Known Alias", "Alias", fuzz_ratio, first_match
    if matched and method == "Fuzzy":
        return "Typo", "Fuzzy", fuzz_ratio, first_match
    return "Manual Review", "Manual", fuzz_ratio, first_match

results = df_final_template.apply(
    lambda row: pd.Series(compute_reason_and_method(row),
                         index=["match_reason", "match_method", "fuzz_ratio", "first_name_match"]),
    axis=1
)

df_final_template[["match_reason", "match_method", "match_score"]] = results[["match_reason", "match_method", "fuzz_ratio"]]
df_final_template["first_name_match"] = results["first_name_match"]

def tier_from_reason(reason):
    mapping = {
        "Exact": "Tier 1 - Exact Match",
        "Accent": "Tier 2 - Accent Match",
        "Known Alias": "Tier 3 - Known Alias",
        "Typo": "Tier 4 - Typo Match",
        "Manual Review": "Tier 5 - Manual Review"
    }
    return mapping.get(reason, "Tier 5 - Manual Review")

df_final_template["confidence_tier"] = df_final_template["match_reason"].map(tier_from_reason)

df_final_template["canonical_name"] = df_final_template["suggested_canonical_name"]
df_final_template["notes"] = ""

df_final_template = df_final_template.sort_values(
    by=["source_record_count", "cluster_id"], ascending=[False, True]
)

# ----------------------------------------------------------------------
# 11. Save Outputs
# ----------------------------------------------------------------------
CANDIDATE_PATH = CONFIG_DIR / "candidate_duplicates.csv"
REVIEW_PATH = CONFIG_DIR / "review_required.csv"

df_final_template.to_csv(CANDIDATE_PATH, index=False)
df_review = df_final_template[
    df_final_template["confidence_tier"].isin(["Tier 4 - Typo Match", "Tier 5 - Manual Review"])
]
df_review.to_csv(REVIEW_PATH, index=False)

print("\n" + "="*70)
print(f"🎉 CLUSTERING COMPLETE: {df_final_template['cluster_id'].nunique()} Player Groups Formed.")
print(f"💾 Full Ledger: {CANDIDATE_PATH}")
print(f"🔍 Review List (Tier 4+): {REVIEW_PATH}")
print("="*70)


📋 Loaded 4 canonical entries with aliases from aliases.yml
⚙️ Ingesting datasets and extracting player records...
📊 Discovered 3314 unique raw name variants.
🤖 Auto-clustering player variants using Union-Find...
🔗 Pre-merging known aliases...
⚖️ Compared 17423 candidate pairs using dual-token blocking.

🎉 CLUSTERING COMPLETE: 3307 Player Groups Formed.
💾 Full Ledger: config/candidate_duplicates.csv
🔍 Review List (Tier 4+): config/review_required.csv


In [344]:
# ============================================================
# GOAT-Net — Cell C: Immutable ID Assignment & Mapping Lock
# ============================================================
import pandas as pd
from datetime import datetime
from IPython.display import display

CANDIDATE_PATH = CONFIG_DIR / "candidate_duplicates.csv"
FINAL_MAPPING_PATH = CONFIG_DIR / "canonical_player_mapping.csv"

if not CANDIDATE_PATH.exists():
    raise FileNotFoundError(f"❌ Candidate file missing at {CANDIDATE_PATH}. Run Cell B first.")

df_candidates = pd.read_csv(CANDIDATE_PATH)

# Ensure strings are clean
df_candidates["source_player_name"] = df_candidates["source_player_name"].astype(str).str.strip()
df_candidates["canonical_name"] = df_candidates["canonical_name"].fillna("").astype(str).str.strip()

# Exclude empty rows
df_valid = df_candidates[df_candidates["canonical_name"] != ""].copy()

# Integrity Check: Ensure 1 source_player_name in a specific dataset maps to only 1 canonical_name
multi_maps = df_valid.groupby(["source_dataset", "source_player_name"])["canonical_name"].nunique()
invalid = multi_maps[multi_maps > 1]
if not invalid.empty:
    raise ValueError(f"❌ Consistency Error: Source player mapped to multiple canonical names:\n{invalid}")

# Load or Initialize Production Database (Append-Only Pattern)
if FINAL_MAPPING_PATH.exists():
    df_existing = pd.read_csv(FINAL_MAPPING_PATH)
    existing_name_to_id = dict(zip(df_existing["canonical_name"], df_existing["canonical_id"]))

    # Calculate next available numeric ID
    existing_nums = (
        df_existing["canonical_id"]
        .str.replace("GOAT", "", regex=False)
        .str.lstrip("0")
        .replace("", "0")
        .astype(int)
    )
    next_id_idx = existing_nums.max() + 1 if not existing_nums.empty else 1
else:
    df_existing = pd.DataFrame()
    existing_name_to_id = {}
    next_id_idx = 1

# Order canonical names by total record count across all sources
player_total_records = (
    df_valid.groupby("canonical_name")["source_record_count"]
    .sum()
    .sort_values(ascending=False)
)

assigned_mapping = {}
for canonical_name in player_total_records.index:
    if canonical_name in existing_name_to_id:
        assigned_mapping[canonical_name] = existing_name_to_id[canonical_name] # Preserve immutable ID
    else:
        assigned_mapping[canonical_name] = f"GOAT{next_id_idx:06d}" # Assign sequential ID
        next_id_idx += 1

df_valid["canonical_id"] = df_valid["canonical_name"].map(assigned_mapping)
df_valid["updated_at"] = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")

# Append new mappings cleanly without overwriting existing history
df_production = pd.concat([df_existing, df_valid], ignore_index=True)
df_production = df_production.drop_duplicates(
    subset=["source_dataset", "source_player_name"],
    keep="last"
)

cols_order = [
    "canonical_id",
    "canonical_name",
    "source_dataset",
    "source_column",
    "source_player_name",
    "source_record_count",
    "confidence_tier",
    "updated_at",
    "notes"
]
df_production = df_production[[c for c in cols_order if c in df_production.columns]]

# Save Production Configuration
df_production.to_csv(FINAL_MAPPING_PATH, index=False)

print("\n" + "="*70)
print("🎉 IDENTITY RESOLUTION COMPLETE: PRODUCTION MAPPING LOCKED")
print("="*70)
print(f"💾 Production File: {FINAL_MAPPING_PATH}")
print(f"📊 Total Mapped Source Entries: {len(df_production)}")
print(f"🔑 Total Unique Canonical Players (GOAT IDs): {df_production['canonical_id'].nunique()}\n")

display(df_production.head(10))


🎉 IDENTITY RESOLUTION COMPLETE: PRODUCTION MAPPING LOCKED
💾 Production File: config/canonical_player_mapping.csv
📊 Total Mapped Source Entries: 6641
🔑 Total Unique Canonical Players (GOAT IDs): 3307



/tmp/ipykernel_966/1226741973.py:64: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  df_valid["updated_at"] = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")


,canonical_id,canonical_name,source_dataset,source_column,source_player_name,source_record_count,confidence_tier,updated_at,notes
6641,GOAT000001,Lionel Messi,statistical_match_stats,common_name,Lionel Messi,538,Tier 1 - Exact Match,2026-08-05 19:06:44,NaN
6642,GOAT000001,Lionel Messi,spatial_passing_centrality,common_name,Lionel Messi,538,Tier 1 - Exact Match,2026-08-05 19:06:44,NaN
6643,GOAT000001,Lionel Messi,spatial_passing_centrality,statsbomb_player,Lionel Andrés Messi Cuccittini,538,Tier 3 - Known Alias,2026-08-05 19:06:44,NaN
6644,GOAT000002,Sergio Busquets i Burgos,spatial_passing_centrality,statsbomb_player,Sergio Busquets i Burgos,390,Tier 1 - Exact Match,2026-08-05 19:06:44,NaN
6645,GOAT000004,Gerard Piqué Bernabéu,spatial_passing_centrality,statsbomb_player,Gerard Piqué Bernabéu,337,Tier 1 - Exact Match,2026-08-05 19:06:44,NaN
6646,GOAT000005,Andrés Iniesta Luján,spatial_passing_centrality,statsbomb_player,Andrés Iniesta Luján,335,Tier 1 - Exact Match,2026-08-05 19:06:44,NaN
6647,GOAT000006,Xavier Hernández Creus,spatial_passing_centrality,statsbomb_player,Xavier Hernández Creus,263,Tier 1 - Exact Match,2026-08-05 19:06:44,NaN
6648,GOAT000007,Víctor Valdés Arribas,spatial_passing_centrality,statsbomb_player,Víctor Valdés Arribas,254,Tier 1 - Exact Match,2026-08-05 19:06:44,NaN
6649,GOAT000008,Jordi Alba Ramos,spatial_passing_centrality,statsbomb_player,Jordi Alba Ramos,247,Tier 1 - Exact Match,2026-08-05 19:06:44,NaN
6650,GOAT000010,Daniel Alves da Silva,spatial_passing_centrality,statsbomb_player,Daniel Alves da Silva,234,Tier 1 - Exact Match,2026-08-05 19:06:44,NaN


In [345]:
# ============================================================
# GOAT-Net — Cell D: Module 1 (Robust Identity Resolution)
# ============================================================
import pandas as pd
import yaml
import json
from pathlib import Path

print("⚙️ [Module 1] Applying Composite-Key Identity Resolution...")

SCHEMA_PATH = CONFIG_DIR / "dataset_schema.yml"
MAPPING_PATH = CONFIG_DIR / "canonical_player_mapping.csv"
VALIDATION_DIR = REPO / "metadata" / "validation"
VALIDATION_DIR.mkdir(parents=True, exist_ok=True)

with open(SCHEMA_PATH, "r") as f:
    schema_config = yaml.safe_load(f)
datasets_schema = schema_config.get("datasets", {})

df_mapping = pd.read_csv(MAPPING_PATH)

# Composite lookups
id_map = {
    (str(row["source_dataset"]).strip(), str(row["source_player_name"]).strip()): row["canonical_id"]
    for _, row in df_mapping.iterrows()
}
name_map = {row["canonical_id"]: row["canonical_name"] for _, row in df_mapping.iterrows()}

mapped_datasets = {}
module1_audit = {"status": "PASSED", "datasets": {}}

for dataset_key, config in datasets_schema.items():
    file_path = REPO / config["path"]
    df = pd.read_parquet(file_path)

    print(f"\n📖 Processing '{dataset_key}'")
    keys = config.get("keys", {})
    unmapped_errors = set()

    # 1. Map Primary Player
    prim_col = keys.get("primary_player")
    if prim_col and prim_col in df.columns:
        clean_names = df[prim_col].astype(str).str.strip()

        # Look up IDs – use pd.NA for missing (string dtype)
        mapped_ids = clean_names.apply(
            lambda n: id_map.get((dataset_key, n), pd.NA) if n not in ["nan", "None", ""] else pd.NA
        )

        # Track failures
        unmapped = clean_names[(mapped_ids.isna()) & (~clean_names.isin(["nan", "None", ""]))].unique()
        for u in unmapped: unmapped_errors.add(f"{prim_col}: '{u}'")

        # Create canonical pillars using Pandas StringDtype
        df["canonical_id"] = mapped_ids.astype("string")
        df["canonical_name"] = df["canonical_id"].map(name_map).astype("string")

    # 2. Map Secondary Players (e.g., passer, receiver) gracefully
    sec_cols = keys.get("secondary_players", [])
    for sec_col in sec_cols:
        if sec_col in df.columns:
            clean_names = df[sec_col].astype(str).str.strip()
            mapped_ids = clean_names.apply(
                lambda n: id_map.get((dataset_key, n), pd.NA) if n not in ["nan", "None", ""] else pd.NA
            )
            df[f"{sec_col}_canonical_id"] = mapped_ids.astype("string")
            df[f"{sec_col}_canonical_name"] = df[f"{sec_col}_canonical_id"].map(name_map).astype("string")

    if unmapped_errors:
        raise ValueError(f"❌ Unmapped primary players found in '{dataset_key}':\n{list(unmapped_errors)[:10]}")

    mapped_datasets[dataset_key] = df
    unique_ids = df["canonical_id"].nunique(dropna=True)
    print(f"   ↳ Mapped {unique_ids} Unique Canonical Players")

    module1_audit["datasets"][dataset_key] = {"unique_canonical_ids": unique_ids}

audit_path = VALIDATION_DIR / "module1_identity.json"
with open(audit_path, "w") as f:
    json.dump(module1_audit, f, indent=2)
print(f"\n🎉 MODULE 1 COMPLETE.")

⚙️ [Module 1] Applying Composite-Key Identity Resolution...

📖 Processing 'statistical_player_summary'
   ↳ Mapped 8 Unique Canonical Players

📖 Processing 'statistical_season_stats'
   ↳ Mapped 8 Unique Canonical Players

📖 Processing 'statistical_match_stats'
   ↳ Mapped 7 Unique Canonical Players

📖 Processing 'spatial_player_summary'
   ↳ Mapped 7 Unique Canonical Players

📖 Processing 'spatial_passing_centrality'
   ↳ Mapped 7 Unique Canonical Players

📖 Processing 'spatial_zone_summary'
   ↳ Mapped 7 Unique Canonical Players

🎉 MODULE 1 COMPLETE.


In [346]:
# =============================================================================
# Diagnostic script: canonical player mapping integrity checks
# Run in Google Colab after uploading/mounting your data
# =============================================================================

import pandas as pd
import re
from pathlib import Path

# -----------------------------------------------------------------------------
# 0. Configuration – set this to your folder containing the CSVs
# -----------------------------------------------------------------------------
# Example for Colab (mount Drive first):
# from google.colab import drive
# drive.mount('/content/drive')
# CONFIG_DIR = Path("/content/drive/MyDrive/your_project_folder")
#
# If files are in the current working directory, use:
# CONFIG_DIR = Path(".")
# -----------------------------------------------------------------------------
CONFIG_DIR = Path(".")   # <--- CHANGE THIS TO YOUR ACTUAL DIRECTORY

# -----------------------------------------------------------------------------
# 1. Check if GOAT000003 exists
# -----------------------------------------------------------------------------
mapping = pd.read_csv("/content/drive/MyDrive/GOAT-Net/config/canonical_player_mapping.csv")

goat3 = mapping[mapping["canonical_id"] == "GOAT000003"]
print(f"Rows with GOAT000003: {len(goat3)}")
display(goat3)   # if run in a notebook cell, this shows a DataFrame

# -----------------------------------------------------------------------------
# 2. Find all missing GOAT IDs
# -----------------------------------------------------------------------------
ids = (
    mapping["canonical_id"]
    .str.extract(r"GOAT(\d+)", expand=False)
    .astype(int)
)

max_id = ids.max()
existing = set(ids)

missing = [
    f"GOAT{i:06d}"
    for i in range(1, max_id + 1)
    if i not in existing
]

print("\nMissing GOAT IDs:")
print(missing)

# -----------------------------------------------------------------------------
# 3. Check who disappeared around GOAT000003
# -----------------------------------------------------------------------------
print("\nFirst 10 rows sorted by canonical_id:")
display(mapping.sort_values("canonical_id").head(10))

# -----------------------------------------------------------------------------
# 4. Compare with previous mapping (if you have an old version)
# -----------------------------------------------------------------------------
# Uncomment and adjust filename if you saved an older mapping:
# old_mapping = pd.read_csv(CONFIG_DIR / "old_canonical_player_mapping.csv")
# removed = set(old_mapping["canonical_name"]) - set(mapping["canonical_name"])
# print("\nRemoved players (present in old, missing in new):")
# print(sorted(removed))

# -----------------------------------------------------------------------------
# 5. Find duplicated canonical names (possible accidental merges)
# -----------------------------------------------------------------------------
dup = (
    mapping.groupby("canonical_name")
    .size()
    .sort_values(ascending=False)
)

print("\nTop 20 duplicated canonical names:")
display(dup.head(20))

# -----------------------------------------------------------------------------
# 6. Check whether cluster count matches canonical name count
# -----------------------------------------------------------------------------
candidate = pd.read_csv("/content/drive/MyDrive/GOAT-Net/config/candidate_duplicates.csv")

n_clusters = candidate["cluster_id"].nunique()
n_names = candidate["canonical_name"].nunique()
print(f"\nUnique clusters: {n_clusters}")
print(f"Unique canonical names: {n_names}")
if n_clusters == n_names:
    print("✅ Cluster count matches canonical name count.")
else:
    print("❌ MISMATCH! These should be equal.")

# -----------------------------------------------------------------------------
# 7. Most useful diagnostic – detailed cluster view
# -----------------------------------------------------------------------------
cluster = (
    candidate.groupby("cluster_id")
    .agg(
        canonical=("canonical_name", "first"),
        variants=("source_player_name", "count"),
        names=("source_player_name", lambda x: list(x))
    )
)

cluster = cluster.sort_values("variants", ascending=False)

print("\nTop 20 clusters by number of variants:")
display(cluster.head(20))

Rows with GOAT000003: 8


,canonical_id,canonical_name,source_dataset,source_column,source_player_name,source_record_count,confidence_tier,updated_at,notes
17,GOAT000003,Neymar,statistical_match_stats,common_name,Neymar,122,Tier 1 - Exact Match,2026-08-05 19:06:44,NaN
18,GOAT000003,Neymar,spatial_passing_centrality,common_name,Neymar,122,Tier 1 - Exact Match,2026-08-05 19:06:44,NaN
19,GOAT000003,Neymar,spatial_passing_centrality,statsbomb_player,Neymar da Silva Santos Junior,122,Tier 3 - Known Alias,2026-08-05 19:06:44,NaN
729,GOAT000003,Neymar,statistical_season_stats,common_name,Neymar,5,Tier 1 - Exact Match,2026-08-05 19:06:44,NaN
2026,GOAT000003,Neymar,statistical_player_summary,common_name,Neymar,1,Tier 1 - Exact Match,2026-08-05 19:06:44,NaN
2027,GOAT000003,Neymar,spatial_player_summary,common_name,Neymar,1,Tier 1 - Exact Match,2026-08-05 19:06:44,NaN
2028,GOAT000003,Neymar,spatial_zone_summary,common_name,Neymar,1,Tier 1 - Exact Match,2026-08-05 19:06:44,NaN
2029,GOAT000003,Neymar,spatial_zone_summary,statsbomb_player,Neymar da Silva Santos Junior,1,Tier 3 - Known Alias,2026-08-05 19:06:44,NaN



Missing GOAT IDs:
[]

First 10 rows sorted by canonical_id:


,canonical_id,canonical_name,source_dataset,source_column,source_player_name,source_record_count,confidence_tier,updated_at,notes
0,GOAT000001,Lionel Messi,statistical_match_stats,common_name,Lionel Messi,538,Tier 1 - Exact Match,2026-08-05 19:06:44,NaN
1,GOAT000001,Lionel Messi,spatial_passing_centrality,common_name,Lionel Messi,538,Tier 1 - Exact Match,2026-08-05 19:06:44,NaN
2,GOAT000001,Lionel Messi,spatial_passing_centrality,statsbomb_player,Lionel Andrés Messi Cuccittini,538,Tier 3 - Known Alias,2026-08-05 19:06:44,NaN
728,GOAT000001,Lionel Messi,statistical_season_stats,common_name,Lionel Messi,5,Tier 1 - Exact Match,2026-08-05 19:06:44,NaN
2020,GOAT000001,Lionel Messi,spatial_zone_summary,statsbomb_player,Lionel Andrés Messi Cuccittini,1,Tier 3 - Known Alias,2026-08-05 19:06:44,NaN
2019,GOAT000001,Lionel Messi,spatial_zone_summary,common_name,Lionel Messi,1,Tier 1 - Exact Match,2026-08-05 19:06:44,NaN
2018,GOAT000001,Lionel Messi,spatial_player_summary,common_name,Lionel Messi,1,Tier 1 - Exact Match,2026-08-05 19:06:44,NaN
2017,GOAT000001,Lionel Messi,statistical_player_summary,common_name,Lionel Messi,1,Tier 1 - Exact Match,2026-08-05 19:06:44,NaN
2021,GOAT000002,Sergio Busquets i Burgos,spatial_zone_summary,statsbomb_player,Sergio Busquets i Burgos,1,Tier 1 - Exact Match,2026-08-05 19:06:44,NaN
3,GOAT000002,Sergio Busquets i Burgos,spatial_passing_centrality,statsbomb_player,Sergio Busquets i Burgos,390,Tier 1 - Exact Match,2026-08-05 19:06:44,NaN



Top 20 duplicated canonical names:


,0
canonical_name,
Neymar,8
Lionel Messi,8
Cristiano Ronaldo,8
Kylian Mbappé,8
Kevin De Bruyne,6
Luka Modrić,6
Robert Lewandowski,6
N'Golo Kanté,4
Alfred John Momar N''Diaye,4



Unique clusters: 3307
Unique canonical names: 3307
✅ Cluster count matches canonical name count.

Top 20 clusters by number of variants:


,canonical,variants,names
cluster_id,,,
GROUP_00018,Cristiano Ronaldo,8,"[Cristiano Ronaldo, Cristiano Ronaldo, Cristia..."
GROUP_00049,Kylian Mbappé,8,"[Kylian Mbappé, Kylian Mbappé, Kylian Mbappé L..."
GROUP_00056,Robert Lewandowski,8,"[Robert Lewandowski, Robert Lewandowski, Rober..."
GROUP_00019,Kevin De Bruyne,8,"[Kevin De Bruyne, Kevin De Bruyne, Kevin De Br..."
GROUP_00010,Luka Modrić,8,"[Luka Modrić, Luka Modrić, Luka Modrić, Luka M..."
GROUP_00001,Lionel Messi,8,"[Lionel Messi, Lionel Messi, Lionel Andrés Mes..."
GROUP_00007,Neymar,8,"[Neymar, Neymar, Neymar da Silva Santos Junior..."
GROUP_00782,Alfred John Momar N''Diaye,4,"[Alfred John Momar N''Diaye, Alfred John Momar..."
GROUP_00182,N'Golo Kanté,4,"[N'Golo Kanté, N''Golo Kanté, N''Golo Kanté, N..."


In [347]:
candidate = pd.read_csv(
    "/content/drive/MyDrive/GOAT-Net/config/candidate_duplicates.csv"
)

for cluster_id, group in candidate.groupby("cluster_id"):

    unique = group["source_player_name"].unique()

    if len(unique) > 1:
       print(f"\n{cluster_id}")
       print("-"*60)
       print("="*80)
       print(cluster_id)
       print(group["canonical_name"].iloc[0])
       print(unique)
       for n in unique:
            print(n)




GROUP_00001
------------------------------------------------------------
GROUP_00001
Lionel Messi
['Lionel Messi' 'Lionel Andrés Messi Cuccittini']
Lionel Messi
Lionel Andrés Messi Cuccittini

GROUP_00007
------------------------------------------------------------
GROUP_00007
Neymar
['Neymar' 'Neymar da Silva Santos Junior']
Neymar
Neymar da Silva Santos Junior

GROUP_00018
------------------------------------------------------------
GROUP_00018
Cristiano Ronaldo
['Cristiano Ronaldo' 'Cristiano Ronaldo dos Santos Aveiro']
Cristiano Ronaldo
Cristiano Ronaldo dos Santos Aveiro

GROUP_00049
------------------------------------------------------------
GROUP_00049
Kylian Mbappé
['Kylian Mbappé' 'Kylian Mbappé Lottin']
Kylian Mbappé
Kylian Mbappé Lottin

GROUP_00182
------------------------------------------------------------
GROUP_00182
N'Golo Kanté
["N'Golo Kanté" "N''Golo Kanté"]
N'Golo Kanté
N''Golo Kanté

GROUP_00435
------------------------------------------------------------
GROUP_0

In [348]:
candidate = pd.read_csv("/content/drive/MyDrive/GOAT-Net/config/candidate_duplicates.csv")

dup = (
    candidate.groupby("canonical_name")["cluster_id"]
    .nunique()
)

dup = dup[dup > 1]

print(dup)

Series([], Name: cluster_id, dtype: int64)


In [349]:
candidate[
    candidate.cluster_id=="GROUP_00570"
][[
    "source_player_name",
    "normalized_name",
    "confidence_tier",
    "match_method"
]]

,source_player_name,normalized_name,confidence_tier,match_method
583,Sergio Sánchez Ortega,sergio sanchez ortega,Tier 1 - Exact Match,Self
2612,Sergio Sánchez Ortega,sergio sanchez ortega,Tier 1 - Exact Match,Self


In [350]:
matched, method, score, _ = match_pair(n1, n2)

if matched:
    print(f"{method:6} {score:3} : {n1}  <-->  {n2}")
    uf.union(i1, i2)

In [351]:
for x in [
    "sergio sanchez ortega",
    "sergio sanchez sanchez",
    "sergio mora sanchez"
]:
    print(x)
    print("canonical:", alias_to_canonical.get(x))
    print("preferred:", alias_to_preferred.get(x))
    print()

sergio sanchez ortega
canonical: None
preferred: None

sergio sanchez sanchez
canonical: None
preferred: None

sergio mora sanchez
canonical: None
preferred: None



In [352]:
print(alias_to_canonical.get("sergio sanchez ortega"))
print(alias_to_canonical.get("sergio sanchez sanchez"))
print(alias_to_canonical.get("sergio mora sanchez"))

for cid, grp in df_variants.groupby("cluster_id"):
    if "sergio mora sanchez" in grp["normalized_name"].values:
        print(grp[["source_player_name", "normalized_name"]])

None
None
None
       source_player_name      normalized_name
2892  Sergio Mora Sánchez  sergio mora sanchez


In [353]:
for name, obj in globals().items():
    if isinstance(obj, pd.DataFrame):
        cols = list(obj.columns)

        if ("canonical_id" in cols) or ("preferred_name" in cols):
            print("\n", name)
            print(cols)


 df
['statsbomb_player', 'attacking', 'defensive', 'midfield', 'canonical_id', 'common_name', 'total_touches', 'defensive_pct', 'midfield_pct', 'attacking_pct', 'canonical_name', 'statsbomb_player_canonical_id', 'statsbomb_player_canonical_name']

 df_valid
['source_dataset', 'source_column', 'source_player_name', 'normalized_name', 'source_record_count', 'cluster_id', 'suggested_canonical_name', 'match_reason', 'match_method', 'match_score', 'first_name_match', 'confidence_tier', 'canonical_name', 'notes', 'canonical_id', 'updated_at']

 df_existing
['canonical_id', 'canonical_name', 'source_dataset', 'source_column', 'source_player_name', 'source_record_count', 'confidence_tier', 'updated_at', 'notes']

 df_production
['canonical_id', 'canonical_name', 'source_dataset', 'source_column', 'source_player_name', 'source_record_count', 'confidence_tier', 'updated_at', 'notes']

 df_mapping
['canonical_id', 'canonical_name', 'source_dataset', 'source_column', 'source_player_name', 'source

In [354]:
for name, obj in globals().items():
    if isinstance(obj, pd.DataFrame):
        print(name)

df
df_extracted
df_template
df_raw
df_variants
df_final_template
df_review
df_candidates
df_valid
df_existing
df_production
df_mapping
mapping
goat3
candidate
cluster
group
grp
results
_80
_106
_128
_144
_151
_152
_153
_154
_155
_156
_166
_173
_174
_175
_185
_192
_193
_194
_205
_212
_213
_214
df_inventory
macro_df
micro_df
df_coverage
_229
_236
_237
_238
df_density
_254
_261
_262
_263
_278
_285
_286
_287
df_volume
df_spatial_summary
df_spatial_zones
df_profile
_301
_308
_309
_310
_324
_331
_332
_333
df_registry
_349


In [355]:
cid = df_variants.loc[
    df_variants["normalized_name"]=="sergio mora sanchez",
    "cluster_id"
].iloc[0]

grp = df_variants[df_variants.cluster_id==cid]

print(grp[["normalized_name"]].sort_values("normalized_name"))

          normalized_name
2892  sergio mora sanchez


In [356]:
df_valid = df_candidates[df_candidates["canonical_name"] != ""].copy()

df_candidates[
    df_candidates["normalized_name"].str.contains("sergio", case=False, na=False)
][[
    "source_player_name",
    "normalized_name",
    "canonical_name",
    "match_method",
    "confidence_tier"
]]

,source_player_name,normalized_name,canonical_name,match_method,confidence_tier
3,Sergio Busquets i Burgos,sergio busquets i burgos,Sergio Busquets i Burgos,Self,Tier 1 - Exact Match
55,Sergio Ramos García,sergio ramos garcia,Sergio Ramos García,Self,Tier 1 - Exact Match
92,Sergio Leonel Agüero del Castillo,sergio leonel aguero del castillo,Sergio Leonel Agüero del Castillo,Self,Tier 1 - Exact Match
155,Sergio Canales Madrazo,sergio canales madrazo,Sergio Canales Madrazo,Self,Tier 1 - Exact Match
157,Sergio Asenjo Andrés,sergio asenjo andres,Sergio Asenjo Andrés,Self,Tier 1 - Exact Match
...,...,...,...,...,...
5927,Sergio Rochet Álvarez,sergio rochet alvarez,Sergio Rochet Álvarez,Self,Tier 1 - Exact Match
5928,Sergio Sánchez Sánchez,sergio sanchez sanchez,Sergio Sánchez Sánchez,Self,Tier 1 - Exact Match
5929,Sergio Sánchez Sánchez,sergio sanchez sanchez,Sergio Sánchez Sánchez,Self,Tier 1 - Exact Match
6218,Valentín Sergio Pachón Mosquero,valentin sergio pachon mosquero,Valentín Sergio Pachón Mosquero,Self,Tier 1 - Exact Match


In [357]:
df_candidates[
    df_candidates["source_player_name"].str.contains("Mora", case=False, na=False)
][[
    "source_player_name",
    "normalized_name",
    "canonical_name",
    "cluster_id",
    "match_method",
    "confidence_tier"
]]




,source_player_name,normalized_name,canonical_name,cluster_id,match_method,confidence_tier
77,José Edmílson Gomes de Moraes,jose edmilson gomes de moraes,José Edmílson Gomes de Moraes,GROUP_00070,Self,Tier 1 - Exact Match
230,Adrián González Morales,adrian gonzalez morales,Adrián González Morales,GROUP_00217,Self,Tier 1 - Exact Match
278,Manuel del Moral Fernández,manuel del moral fernandez,Manuel del Moral Fernández,GROUP_00265,Self,Tier 1 - Exact Match
295,Álvaro Borja Morata Martín,alvaro borja morata martin,Álvaro Borja Morata Martín,GROUP_00282,Self,Tier 1 - Exact Match
332,José Luis Morales Nogales,jose luis morales nogales,José Luis Morales Nogales,GROUP_00319,Self,Tier 1 - Exact Match
492,Celso Borges Mora,celso borges mora,Celso Borges Mora,GROUP_00479,Self,Tier 1 - Exact Match
601,Roberto Torres Morales,roberto torres morales,Roberto Torres Morales,GROUP_00588,Self,Tier 1 - Exact Match
700,Ignacio Cases Mora,ignacio cases mora,Ignacio Cases Mora,GROUP_00683,Self,Tier 1 - Exact Match
1410,Ángel Morales Cuerva,angel morales cuerva,Ángel Morales Cuerva,GROUP_01388,Self,Tier 1 - Exact Match
1568,Antonio Moral Segura,antonio moral segura,Antonio Moral Segura,GROUP_01546,Self,Tier 1 - Exact Match


In [358]:
df_review[
    df_review["source_player_name"].str.contains("Mora", case=False, na=False)
]

,source_dataset,source_column,source_player_name,normalized_name,source_record_count,cluster_id,suggested_canonical_name,match_reason,match_method,match_score,first_name_match,confidence_tier,canonical_name,notes


In [359]:
import pandas as pd
import numpy as np
import yaml
import json
from pathlib import Path

print("⚙️ [Module 1] Applying Composite-Key Identity Resolution...")

# Re-initialize REPO and CONFIG_DIR to ensure correct paths
REPO = Path("/content/drive/MyDrive/GOAT-Net")
CONFIG_DIR = REPO / "config"

SCHEMA_PATH = CONFIG_DIR / "dataset_schema.yml"
MAPPING_PATH = CONFIG_DIR / "canonical_player_mapping.csv"
VALIDATION_DIR = REPO / "metadata" / "validation"
VALIDATION_DIR.mkdir(parents=True, exist_ok=True)

with open(SCHEMA_PATH, "r") as f:
    schema_config = yaml.safe_load(f)
datasets_schema = schema_config.get("datasets", {})

df_mapping = pd.read_csv(MAPPING_PATH)

# Build Composite Lookup: (source_dataset, source_player_name) -> canonical_id
id_map = {
    (str(row["source_dataset"]).strip(), str(row["source_player_name"]).strip()): row["canonical_id"]
    for _, row in df_mapping.iterrows()
}
name_map = {row["canonical_id"]: row["canonical_name"] for _, row in df_mapping.iterrows()}

mapped_datasets = {}
module1_audit = {"status": "PASSED", "datasets": {}}

for dataset_key, config in datasets_schema.items():
    file_path = REPO / config["path"]
    df = pd.read_parquet(file_path)

    print(f"\n📖 Processing '{dataset_key}'")
    keys = config.get("keys", {})
    unmapped_errors = set()

    # 1. Map Primary Player
    prim_col = keys.get("primary_player")
    if prim_col and prim_col in df.columns:
        clean_names = df[prim_col].astype(str).str.strip()

        # Look up IDs – use pd.NA for missing (string dtype)
        mapped_ids = clean_names.apply(
            lambda n: id_map.get((dataset_key, n), pd.NA) if n not in ["nan", "None", ""] else pd.NA
        )

        # Track failures
        unmapped = clean_names[(mapped_ids.isna()) & (~clean_names.isin(["nan", "None", ""]))].unique()
        for u in unmapped: unmapped_errors.add(f"{prim_col}: '{u}'")

        # Create canonical pillars using Pandas StringDtype
        df["canonical_id"] = mapped_ids.astype("string")
        df["canonical_name"] = df["canonical_id"].map(name_map).astype("string")

    # 2. Map Secondary Players (e.g., passer, receiver) gracefully
    sec_cols = keys.get("secondary_players", [])
    for sec_col in sec_cols:
        if sec_col in df.columns:
            clean_names = df[sec_col].astype(str).str.strip()
            mapped_ids = clean_names.apply(
                lambda n: id_map.get((dataset_key, n), pd.NA) if n not in ["nan", "None", ""] else pd.NA
            )
            df[f"{sec_col}_canonical_id"] = mapped_ids.astype("string")
            df[f"{sec_col}_canonical_name"] = df[f"{sec_col}_canonical_id"].map(name_map).astype("string")

    if unmapped_errors:
        raise ValueError(f"❌ Unmapped primary players found in '{dataset_key}':\n{list(unmapped_errors)[:10]}")

    mapped_datasets[dataset_key] = df
    unique_ids = df["canonical_id"].nunique(dropna=True)
    print(f"   ↳ Mapped {unique_ids} Unique Canonical Players")

    module1_audit["datasets"][dataset_key] = {"unique_canonical_ids": unique_ids}

audit_path = VALIDATION_DIR / "module1_identity.json"
with open(audit_path, "w") as f:
    json.dump(module1_audit, f, indent=2)
print(f"\n🎉 MODULE 1 COMPLETE.")

⚙️ [Module 1] Applying Composite-Key Identity Resolution...

📖 Processing 'statistical_player_summary'
   ↳ Mapped 8 Unique Canonical Players

📖 Processing 'statistical_season_stats'
   ↳ Mapped 8 Unique Canonical Players

📖 Processing 'statistical_match_stats'
   ↳ Mapped 7 Unique Canonical Players

📖 Processing 'spatial_player_summary'
   ↳ Mapped 7 Unique Canonical Players

📖 Processing 'spatial_passing_centrality'
   ↳ Mapped 7 Unique Canonical Players

📖 Processing 'spatial_zone_summary'
   ↳ Mapped 7 Unique Canonical Players

🎉 MODULE 1 COMPLETE.


In [360]:
import re

print("⚙️ [Module 2] Standardizing Schemas and Data Types...")

standardized_datasets = {}

def clean_column_name(col_name):
    col = str(col_name).lower().strip()
    col = col.replace("%", "_pct")
    col = re.sub(r"[^a-z0-9_]", "_", col)
    col = re.sub(r"_+", "_", col)
    return col.strip("_")

for dataset_key, df in mapped_datasets.items():
    mem_before = df.memory_usage(deep=True).sum() / (1024**2)

    # 1. Clean names
    df.columns = [clean_column_name(c) for c in df.columns]

    # 2. Convert to intelligent pandas dtypes (string, Int64, Float64)
    df = df.convert_dtypes()

    # 3. Downcast numerics intelligently – keep IDs and names untouched
    for col in df.columns:
        if "id" in col or "name" in col:
            continue
        if pd.api.types.is_integer_dtype(df[col]):
            df[col] = pd.to_numeric(df[col], downcast='integer')
        elif pd.api.types.is_float_dtype(df[col]):
            df[col] = pd.to_numeric(df[col], downcast='float')

    mem_after = df.memory_usage(deep=True).sum() / (1024**2)
    standardized_datasets[dataset_key] = df

    print(f"📖 '{dataset_key}' -> Memory: {mem_before:.2f} MB -> {mem_after:.2f} MB")

print(f"\n🎉 MODULE 2 COMPLETE.")

⚙️ [Module 2] Standardizing Schemas and Data Types...
📖 'statistical_player_summary' -> Memory: 0.00 MB -> 0.00 MB
📖 'statistical_season_stats' -> Memory: 0.02 MB -> 0.02 MB
📖 'statistical_match_stats' -> Memory: 0.19 MB -> 0.17 MB
📖 'spatial_player_summary' -> Memory: 0.00 MB -> 0.00 MB
📖 'spatial_passing_centrality' -> Memory: 7.09 MB -> 7.62 MB
📖 'spatial_zone_summary' -> Memory: 1.28 MB -> 1.33 MB

🎉 MODULE 2 COMPLETE.


In [361]:
# ============================================================
# GOAT-Net — Cell F: Modules 3 & 4 (Inventory & Dataset Density)
# ============================================================
print("⚙️ [Module 3] Building Schema-Driven Competition Inventory...")

INVENTORY_DIR = REPO / "metadata" / "inventory"
COVERAGE_DIR = REPO / "metadata" / "coverage"
INVENTORY_DIR.mkdir(parents=True, exist_ok=True)
COVERAGE_DIR.mkdir(parents=True, exist_ok=True)

inventory_records = []

for dataset_key, df in standardized_datasets.items():
    keys = datasets_schema[dataset_key].get("keys", {})
    comp_col = keys.get("competition")
    season_col = keys.get("season")
    match_col = keys.get("match", keys.get("match_count"))

    if comp_col and season_col and match_col:
        c_comp = clean_column_name(comp_col)
        c_seas = clean_column_name(season_col)
        c_match = clean_column_name(match_col)

        if all(c in df.columns for c in [c_comp, c_seas, c_match, "canonical_id"]):
            grouped = df.groupby(["canonical_id", c_comp, c_seas])[c_match].nunique().reset_index()
            for _, row in grouped.iterrows():
                inventory_records.append({
                    "canonical_id": row["canonical_id"],
                    "competition_name": row[c_comp],
                    "season": row[c_seas],
                    "source_dataset": dataset_key,
                    "matches_recorded": row[c_match]
                })

if inventory_records:
    df_inventory = pd.DataFrame(inventory_records)
    df_inventory = df_inventory.groupby(["canonical_id", "competition_name", "season", "source_dataset"], as_index=False)["matches_recorded"].sum()
    inventory_path = INVENTORY_DIR / "dataset_inventory.csv"
    df_inventory.to_csv(inventory_path, index=False)
    print(f"  ✓ Saved Competition Inventory ({len(df_inventory)} records) to {inventory_path}")
else:
    print("  ⚠️ No competition/season keys found in schema. Skipping Inventory.")

print("\n⚙️ [Module 4] Computing Event Dataset Density...")

density_records = []
macro_df = standardized_datasets.get("statistical_match_stats")
micro_df = standardized_datasets.get("spatial_passing_centrality")

if macro_df is not None and micro_df is not None:
    canonical_ids = macro_df["canonical_id"].dropna().unique()

    for cid in canonical_ids:
        c_name = macro_df.loc[macro_df["canonical_id"] == cid, "canonical_name"].iloc[0]

        macro_match_col = clean_column_name(datasets_schema["statistical_match_stats"]["keys"]["match"])
        basic_matches = macro_df[macro_df["canonical_id"] == cid][macro_match_col].nunique()

        micro_match_col = clean_column_name(datasets_schema["spatial_passing_centrality"]["keys"]["match"])
        event_matches = micro_df[micro_df["canonical_id"] == cid][micro_match_col].nunique()

        density_pct = round((event_matches / basic_matches * 100), 2) if basic_matches > 0 else 0.0

        density_records.append({
            "canonical_id": cid,
            "canonical_name": c_name,
            "macro_dataset_matches": basic_matches,
            "event_dataset_matches": event_matches,
            "event_dataset_density_pct": density_pct
        })

    df_density = pd.DataFrame(density_records)
    if not df_density.empty:
        df_density = df_density.sort_values(by="macro_dataset_matches", ascending=False)
        density_path = COVERAGE_DIR / "career_dataset_density.csv"
        df_density.to_csv(density_path, index=False)
        print(f"  ✓ Computed dataset density for {len(df_density)} panel players.")
        print(f"💾 Density metrics saved to {density_path}")

        from IPython.display import display
        display(df_density.head())
    else:
        print("  ⚠️ No density records to compute. Skipping density metrics.")
else:
    print("  ⚠️ Skipping density math (statistical_match_stats or spatial_passing_centrality not loaded).")

print("\n🎉 MODULES 3 & 4 COMPLETE: Meta-analytics securely generated.")

⚙️ [Module 3] Building Schema-Driven Competition Inventory...
  ⚠️ No competition/season keys found in schema. Skipping Inventory.

⚙️ [Module 4] Computing Event Dataset Density...
  ✓ Computed dataset density for 7 panel players.
💾 Density metrics saved to /content/drive/MyDrive/GOAT-Net/metadata/coverage/career_dataset_density.csv


,canonical_id,canonical_name,macro_dataset_matches,event_dataset_matches,event_dataset_density_pct
3,GOAT000001,Lionel Messi,538,538,100.0
5,GOAT000003,Neymar,122,122,100.0
6,GOAT000009,Cristiano Ronaldo,76,76,100.0
4,GOAT000011,Luka Modrić,71,71,100.0
0,GOAT000019,Kevin De Bruyne,42,42,100.0



🎉 MODULES 3 & 4 COMPLETE: Meta-analytics securely generated.


In [362]:
# ============================================================
# GOAT-Net — Cell G: Modules 5, 6 & 7 (Integration, Validation & Export)
# ============================================================
import json
import pandas as pd
from datetime import datetime

print("⚙️ [Module 5] Executing Career Integration (Style + Volume Merge)...")

INTEGRATED_DIR = REPO / "data" / "processed" / "integrated"
PROVENANCE_DIR = REPO / "metadata" / "provenance"
VALIDATION_DIR = REPO / "metadata" / "validation"

INTEGRATED_DIR.mkdir(parents=True, exist_ok=True)
PROVENANCE_DIR.mkdir(parents=True, exist_ok=True)

# 1. Isolate the Career Summary Datasets
# Volume (FBref)
df_volume = standardized_datasets.get("statistical_player_summary")
# Style (StatsBomb Spatial)
df_spatial_summary = standardized_datasets.get("spatial_player_summary")
df_spatial_zones = standardized_datasets.get("spatial_zone_summary")

if df_volume is None:
    raise ValueError("❌ Critical Failure: 'statistical_player_summary' not found for base volume stats.")

# 2. Base Merge: Start with Volume
# Ensure we keep the canonical_id and canonical_name
df_profile = df_volume.copy()

# 3. Merge Style: Spatial Summary
if df_spatial_summary is not None:
    # Drop canonical_name from right side to avoid _x, _y suffixes
    cols_to_merge = [c for c in df_spatial_summary.columns if c not in ["canonical_name", "common_name", "statsbomb_player"]]
    df_profile = df_profile.merge(
        df_spatial_summary[cols_to_merge],
        on="canonical_id",
        how="left"
    )

# 4. Merge Style: Zone Summary
if df_spatial_zones is not None:
    cols_to_merge = [c for c in df_spatial_zones.columns if c not in ["canonical_name", "common_name", "statsbomb_player"]]
    df_profile = df_profile.merge(
        df_spatial_zones[cols_to_merge],
        on="canonical_id",
        how="left"
    )

# 5. Append Dataset Density Metrics (From Module 4)
if 'df_density' in locals() and not df_density.empty:
    density_cols = ["canonical_id", "event_dataset_density_pct", "macro_dataset_matches", "event_dataset_matches"]
    df_profile = df_profile.merge(
        df_density[density_cols],
        on="canonical_id",
        how="left"
    )

# Clean up any potential duplicate suffixes if they snuck in
df_profile.columns = [c.replace("_x", "").replace("_y", "") for c in df_profile.columns]
df_profile = df_profile.loc[:, ~df_profile.columns.duplicated()]

print(f"  ✓ Merged Career Profile: {len(df_profile)} players, {len(df_profile.columns)} features.")


print("\n⚙️ [Module 6] Running Integrity & Validation Checks...")

integrity_errors = []
integrity_warnings = []

# Rule 1: No Duplicate Canonical IDs
if df_profile["canonical_id"].duplicated().any():
    integrity_errors.append("Duplicate canonical_id detected in final profile.")

# Rule 2: No Null Canonical IDs
if df_profile["canonical_id"].isna().any():
    integrity_errors.append("Null canonical_id detected.")

# Rule 3: Math logical bounds (Check if 'goals' or 'minutes' exist and are valid)
if "goals" in df_profile.columns and (df_profile["goals"] < 0).any():
    integrity_errors.append("Negative goals detected.")
if "minutes" in df_profile.columns and (df_profile["minutes"] < 0).any():
    integrity_errors.append("Negative minutes detected.")

status_str = "PASSED" if not integrity_errors else "FAILED"

validation_report = {
    "timestamp": datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S"),
    "status": status_str,
    "errors": integrity_errors,
    "warnings": integrity_warnings,
    "total_players": len(df_profile),
    "total_features": len(df_profile.columns),
    "pipeline_version": datasets_schema.get("pipeline_version", "3.5.0")
}

val_path = VALIDATION_DIR / "module6_integrity_report.json"
with open(val_path, "w") as f:
    json.dump(validation_report, f, indent=2)

if integrity_errors:
    raise ValueError(f"❌ [Integrity Failed] Pipeline halted due to errors: {integrity_errors}")
print(f"  ✓ Integrity Checks: PASSED. Report saved to {val_path}")


print("\n⚙️ [Module 7] Feature Registry & Provenance Export...")

# 1. Generate Feature Registry
registry_records = []
for col in df_profile.columns:
    if col in ["canonical_id", "canonical_name"]:
        source = "System Identity Layer"
        category = "Identifier"
    elif col in density_cols:
        source = "GOAT-Net Density Engine"
        category = "Metadata"
    elif df_spatial_summary is not None and col in df_spatial_summary.columns:
        source = "StatsBomb Spatial Engine"
        category = "Style (Micro)"
    elif df_spatial_zones is not None and col in df_spatial_zones.columns:
        source = "StatsBomb Zone Engine"
        category = "Style (Micro)"
    else:
        source = "FBref Statistical Engine"
        category = "Volume (Macro)"

    registry_records.append({
        "feature": col,
        "category": category,
        "provenance_source": source
    })

df_registry = pd.DataFrame(registry_records)
registry_path = PROVENANCE_DIR / "feature_registry.csv"
df_registry.to_csv(registry_path, index=False)
print(f"  ✓ Feature Registry generated: {len(df_registry)} features tracked.")

# 2. Append Pipeline Meta-tags
df_profile["pipeline_version"] = datasets_schema.get("pipeline_version", "3.5.0")
df_profile["integrated_at"] = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")

# 3. Export Final Profile
parquet_out = INTEGRATED_DIR / "career_player_profile.parquet"
csv_out = INTEGRATED_DIR / "career_player_profile.csv"

df_profile.to_parquet(parquet_out, index=False)
df_profile.to_csv(csv_out, index=False)

print("\n" + "="*70)
print("🎉 NOTEBOOK 3: CAREER INTEGRATION ENGINE FULLY COMPLETE")
print("="*70)
print(f"💾 Production Parquet: {parquet_out}")
print(f"💾 Data Dictionary: {registry_path}")

from IPython.display import display
display(df_profile.head())

⚙️ [Module 5] Executing Career Integration (Style + Volume Merge)...
  ✓ Merged Career Profile: 8 players, 50 features.

⚙️ [Module 6] Running Integrity & Validation Checks...
  ✓ Integrity Checks: PASSED. Report saved to /content/drive/MyDrive/GOAT-Net/metadata/validation/module6_integrity_report.json

⚙️ [Module 7] Feature Registry & Provenance Export...
  ✓ Feature Registry generated: 50 features tracked.

🎉 NOTEBOOK 3: CAREER INTEGRATION ENGINE FULLY COMPLETE
💾 Production Parquet: /content/drive/MyDrive/GOAT-Net/data/processed/integrated/career_player_profile.parquet
💾 Data Dictionary: /content/drive/MyDrive/GOAT-Net/metadata/provenance/feature_registry.csv


/tmp/ipykernel_966/1709922743.py:88: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S"),
/tmp/ipykernel_966/1709922743.py:140: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  df_profile["integrated_at"] = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")


,canonical_id,common_name,career_minutes,career_goals,career_assists,seasons_played,latest_overall,career_90s,goals_per90,assists_per90,...,defensive_pct,midfield_pct,attacking_pct,statsbomb_player_canonical_id,statsbomb_player_canonical_name,event_dataset_density_pct,macro_dataset_matches,event_dataset_matches,pipeline_version,integrated_at
0,GOAT000019,Kevin De Bruyne,11602,47,68,6,91,128.911118,0.365,0.527,...,8.862483,44.658992,46.478523,GOAT000019,Kevin De Bruyne,100.0,42.0,42.0,3.5.0,2026-08-05 19:06:56
1,GOAT000743,Erling Haaland,10702,125,29,5,<NA>,118.91111,1.051,0.244,...,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,3.5.0,2026-08-05 19:06:56
2,GOAT000056,Robert Lewandowski,16717,174,36,6,90,185.744446,0.937,0.194,...,5.497221,43.545398,50.957382,GOAT000056,Robert Lewandowski,100.0,13.0,13.0,3.5.0,2026-08-05 19:06:56
3,GOAT000033,Kylian Mbappé,14235,162,48,6,91,158.166672,1.024,0.303,...,5.696576,33.825266,60.478157,GOAT000033,Kylian Mbappé,100.0,23.0,23.0,3.5.0,2026-08-05 19:06:56
4,GOAT000001,Lionel Messi,13606,113,73,5,<NA>,151.17778,0.747,0.483,...,1.892914,45.698916,52.408169,GOAT000001,Lionel Messi,100.0,538.0,538.0,3.5.0,2026-08-05 19:06:56


In [363]:
# ============================================================
# GOAT-Net — Cell H: Master GitHub Sync
# ============================================================
import os, subprocess
from pathlib import Path
from google.colab import userdata

REPO = Path("/content/drive/MyDrive/GOAT-Net")
os.chdir(REPO)

def run_git(args, **kwargs):
    return subprocess.run(["git"] + args, cwd=REPO, capture_output=True, text=True, **kwargs)

try:
    TOKEN = userdata.get("GITHUB_TOKEN")
    if not TOKEN: raise ValueError("❌ GITHUB_TOKEN not set in Colab secrets.")

    AUTH_URL = f"https://{TOKEN}@github.com/AtiX-Algo/GOAT-Net.git"
    CLEAN_URL = "https://github.com/AtiX-Algo/GOAT-Net.git"

    run_git(["config", "user.name", "AtiX-Algo"])
    run_git(["config", "user.email", "atix.algo@gmail.com"])
    run_git(["remote", "set-url", "origin", AUTH_URL])

    print("📦 Staging updated config, metadata, and integrated profiles...")
    run_git(["add", "config/"])
    run_git(["add", "metadata/"])
    run_git(["add", "data/processed/integrated/"])

    status = run_git(["status", "--short"]).stdout
    print("\n🔄 Files staged for commit:\n" + (status or "No changes staged."))

    if status:
        run_git(["commit", "-m", "feat: Notebook 3 completed - unified integration engine, strict clustering, full profile merge"])
        run_git(["pull", "--no-rebase", "origin", "main"])
        push = run_git(["push", "-u", "origin", "main", "--force"])

        if push.returncode == 0:
            print("\n✅ Push successful! All integration datasets and configurations are backed up.")
        else:
            print("\n❌ Push failed:\n", push.stderr.replace(TOKEN, "***HIDDEN***"))

    run_git(["remote", "set-url", "origin", CLEAN_URL])

except Exception as e:
    token_str = TOKEN if 'TOKEN' in locals() and TOKEN else "UNKNOWN_TOKEN"
    print(f"\n⚠️ GitHub Sync Skipped: {str(e).replace(token_str, '***HIDDEN***')}")

📦 Staging updated config, metadata, and integrated profiles...

🔄 Files staged for commit:
M  config/canonical_player_mapping.csv
M  data/processed/integrated/career_player_profile.csv
M  data/processed/integrated/career_player_profile.parquet
M  metadata/validation/module6_integrity_report.json
 M notebooks/2_spatial_engine.ipynb
?? notebooks/3_career_integration_engine.ipynb


✅ Push successful! All integration datasets and configurations are backed up.


In [365]:
# ============================================================
# GOAT-Net — Cell I: Final Verification & Testing (FIXED)
# ============================================================
import pandas as pd
from pathlib import Path
from IPython.display import display

REPO = Path("/content/drive/MyDrive/GOAT-Net")
INTEGRATED_DIR = REPO / "data" / "processed" / "integrated"
PROFILE_PATH = INTEGRATED_DIR / "career_player_profile.parquet"
INVENTORY_PATH = REPO / "metadata" / "inventory" / "dataset_inventory.csv"

print("🔍 Loading Integrated Career Profile for testing...")

# Check if profile exists
if not PROFILE_PATH.exists():
    raise FileNotFoundError(f"❌ Profile not found at {PROFILE_PATH}. Run Cell G (Modules 5-7) first.")

df_profile = pd.read_parquet(PROFILE_PATH)

# Load inventory if it exists, otherwise skip
if INVENTORY_PATH.exists():
    df_inventory = pd.read_csv(INVENTORY_PATH)
    print(f"✅ Inventory loaded: {len(df_inventory)} records")
else:
    df_inventory = None
    print("⚠️ Inventory file not found. Competition breakdown will be skipped.")
    print("   (This is normal if Module 3 skipped inventory creation due to missing competition/season keys)")

# 1. Target the Superstars
target_players = ["Lionel Messi", "Cristiano Ronaldo"]
df_test = df_profile[df_profile["canonical_name"].isin(target_players)].copy()

if df_test.empty:
    print("⚠️ Target players not found in the final profile.")
else:
    print(f"✅ Found data for: {', '.join(df_test['canonical_name'].tolist())}")

    # 2. Dynamically find the relevant columns
    id_cols = ["canonical_id", "canonical_name"]
    meta_cols = [c for c in df_profile.columns if "density" in c or "matches" in c]
    volume_cols = [c for c in df_profile.columns if "goal" in c or "assist" in c]

    # Grab a few Style/Spatial metrics
    style_cols = [c for c in df_profile.columns if c not in id_cols + meta_cols + volume_cols][:3]

    display_cols = id_cols + meta_cols + volume_cols + style_cols
    display_cols = [c for c in display_cols if c in df_profile.columns]

    # 3. Display the Master Row Data
    print("\n📊 MASTER PROFILE VERIFICATION (Transposed for easy reading):")
    display(df_test[display_cols].set_index("canonical_name").T)

    # 4. Quick stats check
    print("\n" + "="*70)
    print("📊 KEY METRICS SNAPSHOT")
    print("="*70)

    # Find goal-related columns
    goal_cols = [c for c in df_profile.columns if "goal" in c.lower()]
    assist_cols = [c for c in df_profile.columns if "assist" in c.lower()]
    match_cols = [c for c in df_profile.columns if "match" in c.lower() and "count" not in c.lower()]

    for _, row in df_test.iterrows():
        print(f"\n⚽ {row['canonical_name']} ({row['canonical_id']}):")
        for col in goal_cols[:3]:  # Show first 3 goal metrics
            if col in row:
                print(f"  • {col}: {row[col]}")
        for col in match_cols[:2]:  # Show first 2 match metrics
            if col in row:
                print(f"  • {col}: {row[col]}")

    # 5. Prove Late-Career / Tournament inclusions (only if inventory exists)
    if df_inventory is not None:
        print("\n" + "="*70)
        print("🏆 COMPETITION AGGREGATION PROOF")
        print("="*70)
        for player in target_players:
            canon_id = df_test.loc[df_test["canonical_name"] == player, "canonical_id"]
            if not canon_id.empty:
                cid = canon_id.iloc[0]
                player_inventory = df_inventory[df_inventory["canonical_id"] == cid]

                # Find late-career or international leagues
                late_career_comps = player_inventory[
                    player_inventory["competition_name"].str.contains(
                        "Major League|MLS|Saudi|World Cup|Euro|Copa",
                        case=False,
                        na=False
                    )
                ]

                print(f"\n⚽ {player} (Late-Career & International Tournaments Captured):")
                if not late_career_comps.empty:
                    display(late_career_comps[["competition_name", "season", "matches_recorded"]])
                else:
                    print("  ⚠️ No late-career or tournament competitions found.")
                    print("  This may indicate issues with your FBref extraction in Notebook 1.")
    else:
        print("\n" + "="*70)
        print("🏆 COMPETITION AGGREGATION PROOF")
        print("="*70)
        print("⚠️ Skipped - inventory file not available.")
        print("To generate the inventory, ensure your dataset_schema.yml has")
        print("'competition' and 'season' keys defined for each dataset.")

# Summary
print("\n" + "="*70)
print("📋 VERIFICATION SUMMARY")
print("="*70)
print(f"Total players in profile: {len(df_profile)}")
print(f"Total features: {len(df_profile.columns)}")
print(f"Superstars found: {'✅' if not df_test.empty else '❌'}")
print(f"Inventory available: {'✅' if df_inventory is not None else '⚠️ (Module 3 needs competition/season keys)'}")
print("\n💡 If inventory is missing, check your dataset_schema.yml and re-run Cell F (Modules 3&4).")

🔍 Loading Integrated Career Profile for testing...
⚠️ Inventory file not found. Competition breakdown will be skipped.
   (This is normal if Module 3 skipped inventory creation due to missing competition/season keys)
✅ Found data for: Lionel Messi, Cristiano Ronaldo

📊 MASTER PROFILE VERIFICATION (Transposed for easy reading):


canonical_name,Lionel Messi,Cristiano Ronaldo
canonical_id,GOAT000001,GOAT000009
matches,538,76
event_dataset_density_pct,100.0,100.0
macro_dataset_matches,538.0,76.0
event_dataset_matches,538.0,76.0
career_goals,113,100
career_assists,73,18
goals_per90,0.747,0.79
assists_per90,0.483,0.142
goalcontributions_per90,1.23,0.933



📊 KEY METRICS SNAPSHOT

⚽ Lionel Messi (GOAT000001):
  • career_goals: 113
  • goals_per90: 0.746999979019165
  • goalcontributions_per90: 1.2300000190734863
  • matches: 538
  • macro_dataset_matches: 538.0

⚽ Cristiano Ronaldo (GOAT000009):
  • career_goals: 100
  • goals_per90: 0.7900000214576721
  • goalcontributions_per90: 0.9330000281333923
  • matches: 76
  • macro_dataset_matches: 76.0

🏆 COMPETITION AGGREGATION PROOF
⚠️ Skipped - inventory file not available.
To generate the inventory, ensure your dataset_schema.yml has
'competition' and 'season' keys defined for each dataset.

📋 VERIFICATION SUMMARY
Total players in profile: 8
Total features: 52
Superstars found: ✅
Inventory available: ⚠️ (Module 3 needs competition/season keys)

💡 If inventory is missing, check your dataset_schema.yml and re-run Cell F (Modules 3&4).
